In [ ]:
# init variable
email_lst = ['']
country_names_lst = ['']
base_segment_lst = ['']
actual_segment_lst = ['']
base_segment_mki_lst = ['']
user_id_list = ''
result_df_columns = ['']
tag_names_lst = ['']

In [ ]:
import os
import pandas as pd
import logging
import bcrypt
from clickhouse_driver import Client
import pytz
from datetime import datetime

toolname = 'Операционный VIP дашборд upload'
# stat_tool_table=
# STAT_TOOL_TABLE
# os.environ.get("STAT_TOOL_TABLE")
logging.getLogger("clickhouse_driver").setLevel(logging.ERROR)

#remove_exist files
for file_path in ["VIP_users_dashboard.xlsx", "VIP_users_dashboard.csv"]:
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"{file_path} has been deleted.")
    else:
        print(f"{file_path} does not exist.")

def create_clickhouse_client():
    """
    Подключение к БД по персональным кредам
    """
    return Client(
        host="analyt.1win-eu-west-1.altinity.cloud",
        user=os.environ.get("CH_1WIN_ALTINITY_USER"),
        password=os.environ.get("CH_1WIN_ALTINITY_PASSWORD"),
        database="vip",
        secure=True
    )

client = create_clickhouse_client()

def save_df_to_db(df, table):
    # client = create_clickhouse_client()
    client.execute(f"INSERT INTO {table} VALUES", df.to_dict('records'), types_check=True)    

def hash_password_with_login(login, password):
    salt_part = login[:3].encode('utf-8')  # Take the first 3 letters and encode
    password_with_salt = salt_part + password.encode('utf-8')
    hashed_password = bcrypt.hashpw(password_with_salt, bcrypt.gensalt())
    return hashed_password

def check_password_with_login(login, stored_password, provided_password):
    salt_part = login[:3].encode('utf-8')
    password_with_salt = salt_part + provided_password.encode('utf-8')
    return bcrypt.checkpw(password_with_salt, stored_password)

In [ ]:
login_auth = ""

In [ ]:
password_auth = ""

In [ ]:
df_login = client.query_dataframe(f'''SELECT * FROM vip.vip_dashboard_operations_login t WHERE login='{login_auth}' LIMIT 1''')

if len(df_login)==0:
    print('Login не найден в базе')
    df_to_save = pd.DataFrame({'tool': [toolname], 'datetime':[datetime.now(pytz.timezone('Europe/Moscow'))],
                               'login': [login_auth], 'status':['AuthFailed'], 'extra_options':[None] })
    save_df_to_db(df=df_to_save, table=os.environ.get("STAT_TOOL_TABLE"))    
    raise ValueError('Login не найден в базе')
else:
    stored_password = df_login['password'].loc[0]


password_auth_consensus = check_password_with_login(login=login_auth,
                          stored_password=stored_password.encode('utf-8'),
                          provided_password=password_auth)

if password_auth_consensus:
    print('Login&Pass введён корректно')
    result_df_columns = ['Менеджер_e_mail', 'ID_пользователя', 'Partner_Success', 'Partner_Failed', 'Страна', 'Дата_закрепления', 'Дата_регистрации', 'Дата_получения_последнего_сегмента', 'Дата_последней_активности', 'Склонность_к_оттоку', 'Дата_первого_контакта', 'Статус_последнего_контакта', 'Дата_последнего_контакта', 'ВК', 'Контактный', 'Старый_базовый_сегмент', 'Актуальный_сегмент', 'Базовый_сегмент_МКИ', 'День_рождения', 'Дней_с_последнего_депозита', 'Баланс_игрока_USD', 'Сумма_депозитов', 'Сумма_выводов', 'InOut_за_период', 'Кол_во_неудачных_депозитов', 'Кол_во_депозитов', 'Конверсия_депозитов', 'Сумма_депозитов_за_все_время', 'Сумма_выводов_за_все_время', 'InOut_за_все_время', 'Средний_депозит', 'Сумма_ручных_начислений', 'Сумма_системных_бонусов', 'Сумма_1win_коинов', 'Процент_выданных_бонусов_от_InOut', 'Сумма_кэшбека', 'Сумма_ваучеров', 'Кол_во_ваучеров', 'Сумма_ставок_казино', 'Сумма_ставок_на_спорт', 'Сумма_ставок_в_покере', 'Сумма_выигрышей_казино', 'Сумма_выигрышей_в_спорте','Сумма_выигрышей_в_покере', 'GGR_казино', 'GGR_спорт', 'GGR_покер', 'GGR_общий','Коэффициент_риска', 'МКИ', 'МКИ_30_дней', 'Прирост_МКИ_за_период', 'Дата_последней_смены_пароля', 'Кол_во_смен_пароля_с_3_11', 'Валюта_пользователя', 'Название_тега']
    df_to_save = pd.DataFrame({'tool': [toolname], 'datetime':[datetime.now(pytz.timezone('Europe/Moscow'))],
                               'login': [login_auth], 'status':['AuthPass'], 'extra_options':[None] })
    save_df_to_db(df=df_to_save, table=os.environ.get("STAT_TOOL_TABLE"))
else:
    print('Login&Pass введён не корректно!')
    df_to_save = pd.DataFrame({'tool': [toolname], 'datetime':[datetime.now(pytz.timezone('Europe/Moscow'))],
                               'login': [login_auth], 'status':['AuthFailed'], 'extra_options':[None] })
    save_df_to_db(df=df_to_save, table=os.environ.get("STAT_TOOL_TABLE"))
    raise ValueError('Login&Pass not correct!')
#print('')

In [ ]:
import datetime
date_picker_action = (datetime.date(2025, 01, 12), datetime.date(2025, 01, 16))

In [ ]:
date_start_action = date_picker_action[0].strftime('%Y-%m-%d')
date_end_action = date_picker_action[1].strftime('%Y-%m-%d')

#по требованию Никиты Ко. от 26.11.2024 добавили boris.lyashenko, dmitry.simonov
#в рамках перехода на новую RLS от 03.02.2025 добавили новые логины
#12.02 просьба оставить доступ для zoran.andjelkovic('dmitry.simonov') только для рф
#21.02 добавили dragan.matic в полный доступ по запросу Данила

if login_auth in ('danil.lauman', 'yuliya.pavlova', 'dinara.nurgalieva', 'victor.taranov', 'angela.smith', 'admin', 'boris.lyashenko',
                  'marko.djukic', 'branko.ristic', 'admin', 'dejan.simic','dragan.matic'):
#     query_login_auth=f'''
#                         SELECT distinct manager_email as email
#                         FROM reports.vip_dashboard_operations_v2
#                         '''
    query_login_auth=f'''
                        with managers_id_list as (
                                          SELECT distinct manager_id
                                          FROM reports.vip_dashboard_operations_v2
                                          WHERE tenant_id = 1
                                                )

                        SELECT distinct manager_email as email
                        FROM holistic.vip_segments_daily_enriched_v2
                        WHERE tenant_id = 1
                            and manager_id in ( managers_id_list )
                        '''

elif login_auth == 'zoran.andjelkovic':
        
    query_login_auth=f'''
                        with managers_id_list as (
                                          SELECT distinct manager_id
                                          FROM reports.vip_dashboard_operations_v2
                                          WHERE tenant_id = 1
                                          and country_name = 'Россия'
                                                )

                        SELECT distinct manager_email as email
                        FROM holistic.vip_segments_daily_enriched_v2
                        WHERE tenant_id = 1
                            and manager_id in ( managers_id_list )
                        '''
    
else:
#     query_login_auth=f'''
#                         SELECT distinct email
#                         FROM reports.vw_vip_managers_rls_login
#                         WHERE 1=1
#                         and position(rls_login, '{login_auth}') > 0
#                         '''
    query_login_auth=f'''
      with managers_id_list as (SELECT distinct id
                          FROM reports.vw_vip_managers_rls_login
                          WHERE tenant_id = 1
                            and position(rls_login, '{login_auth}') > 0
                          )

                        SELECT distinct manager_email as email
                        --FROM reports.vip_dashboard_operations_v2
                        FROM holistic.vip_segments_daily_enriched_v2
                        WHERE tenant_id = 1
                            and manager_id in ( managers_id_list )
                        '''    
    
email_df = client.query_dataframe(query_login_auth)
email_lst = email_df['email'].tolist()
# email_lst.insert(0, 'Все email')

country_names_df = client.query_dataframe(f'''
SELECT distinct country_name
FROM reports.vip_dashboard_operations_v2
WHERE tenant_id = 1
order by country_name
''')

country_names_lst = country_names_df['country_name'].tolist()
# country_names_lst.insert(0, 'Все страны')

base_segment_db = client.query_dataframe(f'''
SELECT if(segment_based IS NULL, 'empty', toString(segment_based)) AS base_segment
FROM (SELECT DISTINCT segment_based
      FROM reports.vip_dashboard_operations_v2
      WHERE tenant_id = 1
         )
ORDER BY base_segment
''')
base_segment_lst = base_segment_db['base_segment'].tolist()

actual_segment_db = client.query_dataframe(f'''
SELECT if(segment_actual IS NULL, 'empty', toString(segment_actual)) AS actual_segment
FROM (SELECT DISTINCT segment_actual
      FROM reports.vip_dashboard_operations_v2
      WHERE tenant_id = 1
         )
ORDER BY actual_segment
''')
actual_segment_lst = actual_segment_db['actual_segment'].tolist()


base_segment_mki_db = client.query_dataframe(f'''
SELECT if(segment_based_mki IS NULL, 'empty', toString(segment_based_mki)) AS base_segment_mki
FROM (SELECT DISTINCT segment_based_mki
      FROM reports.vip_dashboard_operations_v2
      WHERE tenant_id = 1
         )
ORDER BY base_segment_mki
''')
base_segment_mki_lst = base_segment_db['base_segment'].tolist()

tag_names_db = client.query_dataframe(f'''
SELECT if(tag_name IS NULL, 'empty', toString(tag_name)) AS tag_name
FROM (select distinct tag_name from holistic.vip_users_meta_tags
where lower(tag_name) not like '%test%' and tenant_id = 1
         )
ORDER BY tag_name
''')
tag_names_lst = tag_names_db['tag_name'].tolist()

# email_lst = ['']
# country_names_lst = ['']
# base_segment_lst = ['']
# actual_segment_lst = ['']
# base_segment_mki_lst = ['']
# user_id_list = ['']
# result_df_columns = ['']

In [ ]:
email_dropdown = []

In [ ]:
country_dropdown = []

In [ ]:
segment_based_dropdown = []

In [ ]:
segment_actual_dropdown = []

In [ ]:
segment_based_mki_dropdown = []

In [ ]:
user_id_list = ""

In [ ]:
tag_dropdown = []

In [ ]:
columns_dropdown = []

In [ ]:
#user_id filter
if user_id_list.strip() == '':
    filter_user_id = ''
else:
    # user_ids = [int(user_id.strip()) for user_id in user_id_list.split(',')]
    user_ids = [int(user_id.strip()) for user_id in user_id_list.split(' ')]
    user_ids_str_list = ','.join(str(user_id) for user_id in list(set(user_ids)))
    filter_user_id = f'and t.user_id in ({user_ids_str_list})'

#email filter
if email_dropdown:
    filter_email_ = "','".join(email_dropdown)
    filter_email = f"and manager_email in ('{filter_email_}')"
else:
    # filter_email='exceptions@1win.xyz', 'partnersVip@1win.xyz', 'NeVip@1win.xyz', 'blockvip@1win.xyz'
    raise ValueError('Выберите email менеджера')

#columns filter
if columns_dropdown:
    pass
else:
    raise ValueError('Выберите колонки для выгрузки')

#country filter
if country_dropdown:
    filter_country_names_ = "','".join(country_dropdown)
    filter_country_names = f"and country_name in ('{filter_country_names_}')"
else:
    filter_country_names=''

#segment_based filter
if segment_based_dropdown:
    filter_segment_based_ = ",".join(segment_based_dropdown)
    filter_segment_based = f"and segment_based in ({filter_segment_based_})"
else:
    filter_segment_based=''

#filter_segment_actual
if segment_actual_dropdown:
    filter_segment_actual_ = ",".join(segment_actual_dropdown)
    filter_segment_actual = f"and segment_actual in ({filter_segment_actual_})"
else:
    filter_segment_actual=''

#filter_segment_based_mki
if segment_based_mki_dropdown:
    filter_segment_based_mki_ = ",".join(segment_based_mki_dropdown)
    filter_segment_based_mki = f"and segment_based_mki in ({filter_segment_based_mki_})"
else:
    filter_segment_based_mki=''

# filter_tag
if tag_dropdown:
    if len(tag_dropdown) == 1:
        filter_tag = f"AND tag_names LIKE '%{tag_dropdown[0]}%'"
    else:
        filter_tag = " AND ".join(f"tag_names LIKE '%{tag}%'" for tag in tag_dropdown)
        filter_tag = f"AND {filter_tag}"
else:
    filter_tag = ''

# --SELECT date, manager_email, user_id, partner_success, partner_failed, country_name, date_vip, reg_date, segment_achieved_date, first_contact_date, last_contact_status, last_contact, contact_flag, deposit_amount, withdrawal_amount, deposit_amount-withdrawal_amount, failed_deposit_count, deposit_count
query_vip_dashboard = '''
with user_id_list as (select distinct user_id
                         from holistic.vip_segments_daily_enriched_report_v2
                         where tenant_id = 1
                           and date = yesterday()
                           and manager_id > 0
                            {filter_email}
                        )

,restore_password_cte as (
select user_id,	toDateTime(max(server_upload_time)) as last_dt_restore_password,
countIf(event_type, server_upload_time >= '2024-11-03') as cnt_change
from holistic.amplitude_1win
where event_type = 'restore_password_success'
and user_id global in ( user_id_list )
GROUP BY 1
)
,currency_users as (
                    SELECT distinct user_id, currency
                    FROM holistic.ma_users_1win
                    where tenant_id = 1
                    and user_id global in ( user_id_list )
                    )
,mki_changes as (
                    select user_id,
                           argMax(max_accumulative_deposits_amount, date) mki,
                           argMin(max_accumulative_deposits_amount, date) mki_old,
                           (mki - mki_old) as mki_changes
                    from holistic.vip_segments_daily_enriched_report_v2
                    where date between subtractDays(toDate('{date_start_action}'), 1) and '{date_end_action}'
                    and tenant_id = 1
                    group by user_id
)                    

SELECT
    `manager_email` as `Менеджер e-mail`,
    t.`user_id` as `ID пользователя`,
    `partner_success` as `Partner Success`,
    `partner_failed` as `Partner Failed`,
    `country_name` as `Страна`,
    `attached_date` as `Дата закрепления`,
    `reg_date` as `Дата регистрации`,
    argMax(`segment_achieved_date`, date) as `Дата получения последнего сегмента`,
    `last_activity_time` as `Дата последней активности`,
    `churn_score` as `Склонность к оттоку`,
    `first_contact_date` as `Дата первого контакта`,--n/n
    `last_contact_status` as `Статус последнего контакта`,
    `last_contact` as `Дата последнего контакта`,
    `withdrawal_manual_control` as `ВК`,
    `contact_flag` as `Контактный`,
    argMax(`segment_based`, date) as `Старый базовый сегмент`,
    argMax(`segment_actual`, date) as `Актуальный сегмент`,
    argMax(`segment_based_mki`, date) as `Базовый сегмент МКИ`,
    `preferences_birthday` as `День рождения`,
    `last_deposit_date` as `Дней с последнего депозита`,
    max(`actual_amount_converted_user_balance`) as `Баланс_игрока_USD`,
    sum(`deposit_amount`) as `Сумма депозитов`,
    sum(`withdrawal_amount`) as `Сумма выводов`,
    sum(`deposit_amount` - `withdrawal_amount`) as `InOut за период`,
    sum(`failed_deposit_count`) as `Кол-во неудачных депозитов`,
    sum(`deposit_count`) as `Кол-во депозитов`,
    if(sum(`deposit_count`+`failed_deposit_count`)=0, 0, sum(`deposit_count`)/sum(`deposit_count`+`failed_deposit_count`)) as `Конверсия депозитов`,
    max(`deposit_sum`) as `Сумма депозитов за все время`,
    max(`withdrawal_sum`) as `Сумма выводов за все время`,
    max(`deposit_sum`) - max(`withdrawal_sum`) as `InOut за все время`,
    if(sum(`deposit_count`)=0, 0, sum(`deposit_amount`)/sum(`deposit_count`)) as `Средний депозит`,
    sum(`hand_bonus`) as `Сумма ручных начислений`,
    sum(`system_bonus`) as `Сумма системных бонусов`,
    sum(`1win_coin_sum`) as `Сумма 1win коинов`,
    if(sum(`deposit_amount` - `withdrawal_amount`)=0,0,sum(`system_bonus`+`voucher_sum`+`1win_coin_sum`) / sum(`deposit_amount` - `withdrawal_amount`)) as `Процент выданных бонусов от InOut`,
    sum(`cashback`) as `Сумма кэшбека`,
    sum(`voucher_sum`) as `Сумма ваучеров`,
    sum(`voucher_count`) as `Кол-во ваучеров`,
    sum(`casino_bets`) as `Сумма ставок казино`,
    sum(`sport_bets`) as `Сумма ставок на спорт`,
    sum(`poker_bets`) as `Сумма ставок в покере`,
    sum(`casino_win`) as `Сумма выигрышей казино`,
    sum(`sport_wins`) as `Сумма выигрышей в спорте`,
    sum(`poker_win`) as `Сумма выигрышей в покере`,
    `Сумма ставок казино` - `Сумма выигрышей казино` as `GGR казино`,
    `Сумма ставок на спорт` - `Сумма выигрышей в спорте` as `GGR спорт`,
    `Сумма ставок в покере` - `Сумма выигрышей в покере` as `GGR покер`,
    `GGR казино` + `GGR спорт` + `GGR покер` as `GGR общий`,
    max(`user_risk_coefficient`) as `Коэффициент риска`,
    max(`max_accumulative_deposits_amount`) as `МКИ`,
    max(`max_accumulative_deposit_amount_w30days`) as `МКИ 30 дней`,
    max(mki_changes.mki_changes) as `Прирост_МКИ_за_период`,
    restore_password_cte.last_dt_restore_password as `Дата_последней_смены_пароля`,
    restore_password_cte.cnt_change as `Кол_во_смен_пароля_с_3_11`,
    currency_users.currency as `Валюта_пользователя`,
    tag_names as `Название тега`
FROM reports.vip_dashboard_operations_v2 t
left join restore_password_cte using user_id
left join currency_users on toInt64(currency_users.user_id)= t.user_id
left join mki_changes on toInt64(mki_changes.user_id) = t.user_id
left join (select user_id, arrayStringConcat(array_agg(tag_name), ', ') AS tag_names
from holistic.vip_users_meta_tags_users t1
         join (select id, tag_name, tenant_id
               from holistic.vip_users_meta_tags
               where lower(tag_name) not like '%test%') t2
              on t1.tag_id = t2.id and t1.tenant_id = t2.tenant_id
where t1.tenant_id = 1
group by 1) tags 
on t.user_id::int = tags.user_id::int

WHERE tenant_id = 1
and date between '{date_start_action}' and '{date_end_action}'
{filter_email}
{filter_country_names}
{filter_segment_based}
{filter_segment_actual}
{filter_segment_based_mki}
{filter_user_id}
{filter_tag}
group by `manager_id`,`manager_email`,t.`user_id`,`partner_success`,`partner_failed`,`country_name`,`attached_date`,`reg_date`,`last_activity_time`,`churn_score`,`first_contact_date`,`last_contact_status`,`last_contact`,`withdrawal_manual_control`,`contact_flag`,`preferences_birthday`,`last_deposit_date`, restore_password_cte.last_dt_restore_password, restore_password_cte.cnt_change, currency_users.currency, `tag_names`
--LIMIT 501
'''

query_vip_dashboard_ = query_vip_dashboard.format(date_start_action=date_start_action,
                                                  date_end_action=date_end_action,
                                                  filter_email = filter_email,
                                                  filter_country_names = filter_country_names,
                                                  filter_segment_based = filter_segment_based,
                                                  filter_segment_actual = filter_segment_actual,
                                                  filter_segment_based_mki = filter_segment_based_mki,
                                                  filter_user_id = filter_user_id,
                                                  filter_tag = filter_tag
                                                  )

result_df = client.query_dataframe(query_vip_dashboard_)

selected_columns = ['Склонность_к_оттоку','Баланс_игрока_USD','Сумма_депозитов', 'Сумма_выводов', 'InOut_за_период', 
                    'Кол_во_неудачных_депозитов','Кол_во_депозитов', 'Конверсия_депозитов', 'Сумма_депозитов_за_все_время',
                    'Сумма_выводов_за_все_время', 'InOut_за_все_время', 'Средний_депозит', 'Сумма_ручных_начислений', 
                    'Сумма_системных_бонусов', 'Сумма_1win_коинов', 'Процент_выданных_бонусов_от_InOut', 'Сумма_кэшбека',
                    'Сумма_ваучеров', 'Кол_во_ваучеров', 'Сумма_ставок_казино', 'Сумма_ставок_на_спорт',
                    'Сумма_ставок_в_покере','Сумма_выигрышей_казино', 'Сумма_выигрышей_в_спорте', 'Сумма_выигрышей_в_покере',
                    'GGR_казино', 'GGR_спорт', 'GGR_покер', 'GGR_общий',
                    'Коэффициент_риска', 'МКИ', 'МКИ_30_дней', 'Прирост_МКИ_за_период']
# result_df[selected_columns] = result_df[selected_columns].applymap(lambda x: f"{x:.2f}")
result_df[selected_columns] = result_df[selected_columns].applymap(lambda x: f"{x:.2f}".replace('.', ','))

try:
    result_df[columns_dropdown].to_csv("VIP_users_dashboard.csv", sep=';', index=None)
    # result_df.to_excel("VIP_users_dashboard.xlsx", index=None)
except:
    print('Возникла ошибка c экспортом файла.')

display(result_df.head())
#display(result_df[columns_dropdown].head())

# result_df_columns = result_df.columns
# pd.DataFrame()
# print('')